# MeIA 2025 — Fine-tuning de BETO para polaridad de reseñas turísticas

**Equipo:** Moodhacker4Neurona · **Score leaderboard:** 0.5889

Clasificación de polaridad 1–5 sobre reseñas turísticas mexicanas del reto MeIA 2025 (Macroentrenamiento en Inteligencia Artificial, UNAM).

Pipeline: enriquecimiento de texto con metadatos → sobremuestreo de la clase minoritaria → fine-tuning de [BETO](https://huggingface.co/dccuchile/bert-base-spanish-wwm-uncased) con las primeras 6 capas congeladas, dropout alto y early stopping.

> Ejecutar con GPU (Colab: *Entorno de ejecución → Cambiar tipo → GPU*).

In [ ]:
%pip install -U -q transformers datasets accelerate sentencepiece scikit-learn openpyxl

## 1. Configuración

In [ ]:
MODEL_NAME   = "dccuchile/bert-base-spanish-wwm-uncased"   # BETO
MAX_LENGTH   = 256
BATCH_SIZE   = 16
NUM_EPOCHS   = 8
LR           = 2e-5
WEIGHT_DECAY = 0.1
WARMUP_RATIO = 0.1
FROZEN_LAYERS = 6      # capas del encoder a congelar
SEED         = 42

# Rutas de los datos del reto (ajustar según el entorno)
TRAIN_XLSX = "MeIA_2025_train.xlsx"
TEST_XLSX  = "MeIA_2025_test_wo_labels.xlsx"

## 2. Carga de datos y enriquecimiento de texto

Cada reseña trae región, ciudad y tipo de servicio. Anteponerlos como pseudo-tokens permite que la atención condicione la polaridad al contexto (un 3★ de hotel no se escribe igual que un 3★ de restaurante).

In [ ]:
import numpy as np
import pandas as pd

df_train = pd.read_excel(TRAIN_XLSX)
df_test  = pd.read_excel(TEST_XLSX)

def enriquecer(fila):
    return f"[REGION: {fila.Region}] [TOWN: {fila.Town}] [SERVICE: {fila.Type}] {fila.Review}"

df_train["text"] = df_train.apply(enriquecer, axis=1)
df_test["text"]  = df_test.apply(enriquecer, axis=1)
df_train["label"] = df_train["Polarity"].astype(int) - 1

print(df_train.shape, df_test.shape)
df_train["Polarity"].value_counts().sort_index()

## 3. Balanceo y partición estratificada

In [ ]:
from sklearn.model_selection import train_test_split

# Sobremuestreo simple de la clase minoritaria hasta la segunda más pequeña
counts   = df_train["label"].value_counts()
minority = counts.idxmin()
extra    = int(counts.drop(index=minority).min() - counts[minority])
if extra > 0:
    sampled  = df_train[df_train.label == minority].sample(extra, replace=True, random_state=SEED)
    df_bal   = pd.concat([df_train, sampled], ignore_index=True).sample(frac=1, random_state=SEED)
else:
    df_bal   = df_train

train_df, val_df = train_test_split(
    df_bal[["text", "label"]], test_size=0.1, stratify=df_bal.label, random_state=SEED
)
len(train_df), len(val_df)

## 4. Tokenización

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df, preserve_index=False).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df,   preserve_index=False).map(tokenize, batched=True)

## 5. Modelo: BETO con regularización fuerte

Con ~5k ejemplos, el fine-tuning completo sobreajusta en 2 épocas. Congelar las primeras 6 capas conserva la morfología general del español y reduce los parámetros entrenables; el dropout alto hace el resto.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    hidden_dropout_prob=0.4,
    attention_probs_dropout_prob=0.3,
    id2label={i: str(i + 1) for i in range(5)},
    label2id={str(i + 1): i for i in range(5)},
)

for name, param in model.bert.named_parameters():
    if name.startswith("encoder.layer") and int(name.split(".")[2]) < FROZEN_LAYERS:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {trainable/1e6:.1f}M")

## 6. Entrenamiento con early stopping

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments

total_steps  = (len(train_ds) // (BATCH_SIZE * 2)) * NUM_EPOCHS

args = TrainingArguments(
    output_dir="results",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=2,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=int(total_steps * WARMUP_RATIO),
    lr_scheduler_type="cosine",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    optim="adamw_torch",
)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "f1_macro": f1_score(p.label_ids, preds, average="macro"),
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
trainer.evaluate()

## 7. Predicción y archivo de envío oficial

In [ ]:
test_ds = Dataset.from_dict({"text": df_test.text.tolist()}).map(tokenize, batched=True)
preds   = np.argmax(trainer.predict(test_ds).predictions, axis=1) + 1

with open("MeIA-BETO.Run.txt", "w") as f:
    for i, p in enumerate(preds):
        f.write(f"MeIA {i} {p}
")

print("Archivo de envío escrito: MeIA-BETO.Run.txt")

## Resultado

Esta configuración obtuvo **0.5889** en el leaderboard del reto MeIA 2025 (equipo *Moodhacker4Neurona*). El archivo de predicciones original está en `reports/runs/MeIA-Moodhacker4neurona.Run.txt`.